# 论文结果闭合入口

该 Notebook 用于在 Colab 中完成当前论文运行层级的结果闭合: 挂载 Google Drive、拉取当前仓库、从 ``SLM_WM_DRIVE_RESULT_ROOT`` 物化前序结果包、重建 attack matrix、external baseline formal import、internal ablation、fixed-FPR 共同协议记录, 并把完整结果包写回 Google Drive。

运行依赖: 必须等待本次要纳入完整结果包的主方法、攻击、dataset-level 质量、method-faithful external baseline 与 official reference 结果包都已经写入 Drive。该入口不需要 GPU, 可以在 CPU Colab 会话或本地同步目录中运行。

Notebook 只调度 `scripts/` 中的 repository commands, 不直接手写正式 records、tables、figures 或 reports。

## 运行前准备

1. 确认前序方法主流程、两类攻击闭环、dataset-level 质量、external baseline 和 official reference Notebook 已把结果包写入 ``SLM_WM_DRIVE_RESULT_ROOT`/`。
2. 该入口不需要 GPU, 但需要能够访问 Google Drive 或已经同步的本地结果包目录。
3. 如果本次需要干净复盘, 应先清理 ``SLM_WM_DRIVE_RESULT_ROOT`/` 后重新运行前序 Notebook, 再运行本入口。
4. 输出完整结果包默认写入 ``SLM_WM_DRIVE_RESULT_ROOT` 下的 `complete_result_package``。

In [ ]:
SLM_WM_PAPER_RUN_NAME = "probe_paper"

import os

from google.colab import drive

drive.mount('/content/drive')
# 可切换为 "probe_paper"、"pilot_paper" 或 "full_paper"。
os.environ["SLM_WM_PAPER_RUN_NAME"] = SLM_WM_PAPER_RUN_NAME


In [ ]:
import os
import re
import subprocess
import sys
from pathlib import Path

repository_url = os.environ.get("SLM_WM_REPOSITORY_URL", "https://github.com/RICHAAARC/SLM-WM.git")
repository_commit = os.environ.get("SLM_WM_REPOSITORY_COMMIT", "")
if re.fullmatch(r"[0-9a-f]{40}", repository_commit) is None:
    raise ValueError("正式运行前必须通过 SLM_WM_REPOSITORY_COMMIT 提供精确40位小写 Git SHA")
workspace_dir = Path(os.environ.get("SLM_WM_WORKSPACE_DIR", "/content/slm_wm_repository"))
workspace_dir.parent.mkdir(parents=True, exist_ok=True)
if workspace_dir.exists() and not (workspace_dir / ".git").is_dir():
    raise RuntimeError("SLM_WM_WORKSPACE_DIR 已存在但不是 Git 仓库")
if not (workspace_dir / ".git").is_dir():
    subprocess.run(["git", "clone", repository_url, str(workspace_dir)], check=True)

status_before_checkout = subprocess.run(
    ["git", "status", "--porcelain=v1", "--untracked-files=all"],
    cwd=workspace_dir,
    check=True,
    capture_output=True,
    text=True,
).stdout
if status_before_checkout:
    raise RuntimeError("复用 Colab 工作区前必须先清理 Git 工作树")
subprocess.run(["git", "fetch", "origin", "--tags", "--prune"], cwd=workspace_dir, check=True)
subprocess.run(["git", "rev-parse", "--verify", repository_commit + "^{commit}"], cwd=workspace_dir, check=True)
subprocess.run(["git", "checkout", "--detach", repository_commit], cwd=workspace_dir, check=True)

os.chdir(workspace_dir)
if str(workspace_dir) not in sys.path:
    sys.path.insert(0, str(workspace_dir))
from paper_workflow.colab_utils.formal_execution import verify_and_publish_formal_execution

formal_execution_lock = verify_and_publish_formal_execution(workspace_dir, repository_commit)
print({"workspace_dir": str(workspace_dir), "formal_execution_lock": formal_execution_lock})

from paper_workflow.colab_utils.paper_run_environment import configure_paper_run_environment
paper_run_environment = configure_paper_run_environment(
    workflow_name="paper_result_closure",
)
print(paper_run_environment)


In [ ]:
from pathlib import Path

package_search_root = Path(os.environ['SLM_WM_PAPER_RUN_PACKAGE_SEARCH_ROOT'])
assert package_search_root.exists(), f'未找到当前论文运行层级的 Google Drive 结果目录: {package_search_root}'
zip_count_by_dir = {
    child.name: len(list(child.glob('*.zip')))
    for child in sorted(package_search_root.iterdir())
    if child.is_dir()
}
print({'package_search_root': str(package_search_root), 'zip_count_by_dir': zip_count_by_dir})


In [ ]:
import os

from paper_workflow.colab_utils.paper_result_closure import run_paper_result_closure_commands

paper_result_closure_summary = run_paper_result_closure_commands(
    package_search_root=os.environ['SLM_WM_PAPER_RUN_PACKAGE_SEARCH_ROOT'],
    complete_drive_output_dir=os.environ['SLM_WM_PAPER_RUN_COMPLETE_DRIVE_OUTPUT_DIR'],
    paper_run_name=os.environ['SLM_WM_PAPER_RUN_NAME'],
)
paper_result_closure_summary


In [ ]:
paper_result_closure_summary
